<h1>ColumnTransformer</h1>
<p>The <b>ColumnTransformer</b> (from scikit-learn) is an incredibly powerful architectural tool. Instead of manually breaking your dataset apart, you hand the ColumnTransformer your entire raw dataset at once

It acts like a traffic cop. You give it a set of rules, and it automatically routes specific columns to the correct preprocessing tools, runs all the transformations simultaneously, and perfectly stitches the final matrix back together for you in one single step</p>

<h2>How the Blueprint Works</h2>
<p>When you build a <b>ColumnTransformer</b>, you provide it with a list of instructions. Each instruction is a simple tuple (a collection of three items in parentheses) that tells the transformer exactly what to do:</p>

<ul>
    <li><b>The Name:</b> A simple nickname you make up for the step (e.g., 'scale_numbers')</li>
    <li><b>The Tool:</b> The actual scikit-learn tool you want to use (e.g., StandardScaler())</li>
    <li><b>The Columns:</b> A list of the specific columns in your dataset you want that tool applied to</li>
</ul>

In [13]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv('covid_toy.csv')
df.sample(5)

,age,gender,fever,cough,city,has_covid
63,10,Male,100.0,Mild,Bangalore,No
83,17,Female,104.0,Mild,Kolkata,No
56,71,Male,NaN,Strong,Kolkata,No
89,46,Male,103.0,Strong,Bangalore,No
14,51,Male,104.0,Mild,Bangalore,No


In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [12]:
X = df.drop(columns = ['has_covid'])
y = df['has_covid']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [22]:
cough_hierarchy = ['Mild','Strong']

transformer = ColumnTransformer(transformers=[
    ('imputer', SimpleImputer(),['fever']),
    ('ordinal_encoded', OrdinalEncoder(categories=[cough_hierarchy]),['cough']),
    ('nominal_enoded',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder = 'passthrough')

transformer.set_output(transform = 'pandas')

ColumnTransformer(remainder='passthrough',
                  transformers=[('imputer', SimpleImputer(), ['fever']),
                                ('ordinal_encoded',
                                 OrdinalEncoder(categories=[['Mild',
                                                             'Strong']]),
                                 ['cough']),
                                ('nominal_enoded',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['gender', 'city'])])

<p><b>The Magic Parameter: remainder='passthrough'</b></br>By default, if you don't explicitly list a column inside the transformers list, the ColumnTransformer will completely delete it from your final dataset.

If you have columns that don't need any scaling or encoding (like a column of 1s and 0s that is already perfectly formatted), you set remainder='passthrough'. This tells the traffic cop: "If I didn't give you instructions for a column, just let it pass through into the final dataset untouched."</p>

In [23]:
transformer.fit_transform(X_train).sample(5)

,imputer__fever,ordinal_encoded__cough,nominal_enoded__gender_Male,nominal_enoded__city_Delhi,nominal_enoded__city_Kolkata,nominal_enoded__city_Mumbai,remainder__age
42,100.0,0.0,1.0,1.0,0.0,0.0,27
74,104.0,1.0,0.0,1.0,0.0,0.0,34
51,100.0,1.0,0.0,0.0,1.0,0.0,11
84,98.0,1.0,0.0,0.0,0.0,1.0,69
6,101.0,1.0,1.0,0.0,0.0,0.0,14
